In [ ]:
import os
from pathlib import Path

# ============================================================
# RUTAS
# ============================================================
PATH_MUNICIPIOS  = "../data/Municipios_Diciembre_2025_shp/Munpio.shp"
PATH_NDVI        = "../data/landsat/landsat_ndvi/"        # archivos ndvi_XXXXX.tif
PATH_DEM         = "../data/dem/dem_srtm/"       # archivos dem_XXXXX.tif
PATH_HIDRO       = "../data/hidrografia/"    # archivos XXXXX_municipio.gpkg
PATH_VIAS_OSM        = "../data/red_vial_osm/"
PATH_POBLADOS    = "../data/centros_poblados/"
PATH_DILIGENCIAS = "../data/Processed/full_MIG.geojson"
PATH_OUTPUT      = "../data/Processed/"

os.makedirs(PATH_OUTPUT, exist_ok=True)

RESOLUCION = 30  # metros
CRS = "EPSG:4326"

In [ ]:

import geopandas as gpd
import pandas as pd
import json

import rasterio

# ============================================================
# CARGAR DATOS BASE
# ============================================================
print("Cargando datos base...")

municipios = gpd.read_file(PATH_MUNICIPIOS)
municipios['MpCodigo'] = municipios['MpCodigo'].astype(str).str.zfill(5)
municipios = municipios.to_crs(CRS)

    
with open(PATH_DILIGENCIAS) as f:
    dilig = json.load(f)
props = [feat['properties'] for feat in dilig['features']]
coords = [feat['geometry']['coordinates'] for feat in dilig['features']]
df_dilig = pd.DataFrame(props)
df_dilig['lon'] = [c[0] for c in coords]
df_dilig['lat'] = [c[1] for c in coords]
df_dilig['Cod_Mun'] = df_dilig['Cod_Mun'].astype(str).str.zfill(5)

gdf_dilig = gpd.GeoDataFrame(
    df_dilig,
    geometry=gpd.points_from_xy(df_dilig['lon'], df_dilig['lat']),
    crs=CRS
)

print(f"  ✓ {len(municipios)} municipios")
print(f"  ✓ {len(gdf_dilig)} diligencias")


In [ ]:
print("Años de diligencias disponibles:")
años = pd.to_numeric(gdf_dilig['Año de la diligencia'], errors='coerce')
print(años.value_counts().sort_index())

print(f"\nRango: {años.min():.0f} - {años.max():.0f}")
print(f"Mediana: {años.median():.0f}")

In [ ]:
PATH_PROCESSED="../data/Processed/"

In [ ]:
import json
import pandas as pd

with open(PATH_PROCESSED+"full_MIG.geojson") as f:
    data = json.load(f)

props = [f['properties'] for f in data['features']]
df = pd.DataFrame(props)

print(f"Total registros: {len(df)}")
print(f"Columnas: {df.columns.tolist()}")
print(f"\nPrimeras filas lat/long:")
print(df[['lat', 'long', 'lat_dd', 'lon_dd']].head(5))

In [ ]:
import re
import numpy as np

def parsear_coordenada(valor):
    """
    Convierte cualquier formato de coordenada a decimal.
    Maneja: decimal, GMS con símbolos, GMS concatenado
    """
    if valor is None or str(valor).strip() in ['', 'SD', 'nan']:
        return None
    
    valor = str(valor).strip()
    
    # Ya es decimal
    try:
        val = float(valor)
        return val
    except:
        pass
    
    # Formato GMS: N9°44'17.6" o 9°44'17.6"
    patron_gms = r"([NSns]?)\s*(\d+)[°]\s*(\d+)['']\s*([\d,.]+)[\"″]?\s*([EWew]?)"
    match = re.search(patron_gms, valor)
    if match:
        hemisferio_pre = match.group(1).upper()
        grados = float(match.group(2))
        minutos = float(match.group(3))
        segundos = float(match.group(4).replace(',', '.'))
        hemisferio_post = match.group(5).upper()
        
        decimal = grados + minutos/60 + segundos/3600
        
        hemisferio = hemisferio_pre or hemisferio_post
        if hemisferio in ['S', 'W']:
            decimal = -decimal
        return decimal
    
    return None

def corregir_fila(row):
    """Determina las coordenadas correctas para una fila."""
    lat_orig = parsear_coordenada(row.get('lat'))
    lon_orig = parsear_coordenada(row.get('long'))
    
    # Rango Colombia
    def es_lat_col(v): return v is not None and -5 <= v <= 13
    def es_lon_col(v): return v is not None and -80 <= v <= -66
    
    # Caso 1: lat y long originales son correctos
    if es_lat_col(lat_orig) and es_lon_col(lon_orig):
        return lat_orig, lon_orig
    
    # Caso 2: lat y long están invertidos
    if es_lon_col(lat_orig) and es_lat_col(lon_orig):
        return lon_orig, lat_orig
    
    # Caso 3: intentar con lat_dd y lon_dd
    try:
        lat_dd = float(row.get('lat_dd', np.nan))
        lon_dd = float(row.get('lon_dd', np.nan))
        if es_lat_col(lat_dd) and es_lon_col(lon_dd):
            return lat_dd, lon_dd
        if es_lon_col(lat_dd) and es_lat_col(lon_dd):
            return lon_dd, lat_dd
    except:
        pass
    
    return None, None

# Aplicar a todos los registros
print("Reconstruyendo coordenadas...")
lats, lons = [], []

for _, row in df.iterrows():
    lat, lon = corregir_fila(row)
    lats.append(lat)
    lons.append(lon)

df['lat_corr'] = lats
df['lon_corr'] = lons

validos = df['lat_corr'].notna()
print(f"Total registros: {len(df)}")
print(f"Coordenadas válidas: {validos.sum()}")
print(f"Sin coordenadas válidas: {(~validos).sum()}")
print(f"\nRango lat: {df.loc[validos,'lat_corr'].min():.3f} a {df.loc[validos,'lat_corr'].max():.3f}")
print(f"Rango lon: {df.loc[validos,'lon_corr'].min():.3f} a {df.loc[validos,'lon_corr'].max():.3f}")

In [ ]:
import geopandas as gpd

# Filtrar solo registros con coordenadas válidas
df_clean = df[df['lat_corr'].notna()].copy()

# Reconstruir GeoDataFrame
gdf_dilig = gpd.GeoDataFrame(
    df_clean,
    geometry=gpd.points_from_xy(df_clean['lon_corr'], df_clean['lat_corr']),
    crs="EPSG:4326"
)

# Usar lon_corr y lat_corr como columnas de referencia
gdf_dilig['lon'] = gdf_dilig['lon_corr']
gdf_dilig['lat'] = gdf_dilig['lat_corr']

print(f"GeoDataFrame reconstruido: {len(gdf_dilig)} registros")
print(f"Presencias (label=1): {(gdf_dilig['label']==1.0).sum()}")
print(f"Ausencias (label=0): {(gdf_dilig['label']==0.0).sum()}")

In [ ]:
df_dilig = gdf_dilig.copy()

In [ ]:
# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def raster_distancia(gdf_lineas, transform, shape):
    if len(gdf_lineas) == 0:
        return np.full(shape, np.nan)
    try:
        burned = rasterize(
            [(geom, 1) for geom in gdf_lineas.geometry 
             if geom is not None and not geom.is_empty],
            out_shape=shape,
            transform=transform,
            fill=0,
            dtype='uint8'
        )
        dist = distance_transform_edt(1 - burned) * RESOLUCION
        return dist
    except:
        return np.full(shape, np.nan)

def derivar_pendiente(dem_array):
    """Calcula pendiente en grados a partir del DEM."""
    dy, dx = np.gradient(dem_array, RESOLUCION)
    pendiente = np.degrees(np.arctan(np.sqrt(dx**2 + dy**2)))
    return pendiente

def cargar_raster_municipio(path_carpeta, codigo, prefijo):
    """Carga raster de un municipio específico."""
    # Buscar archivo con el código
    carpeta = Path(path_carpeta)
    archivos = list(carpeta.glob(f"*{codigo}*.tif"))
    if not archivos:
        return None, None
    with rasterio.open(archivos[0]) as src:
        arr = src.read(1).astype(float)
        arr[arr == src.nodata] = np.nan
        return arr, src.transform



In [ ]:
from rasterio.transform import from_bounds
from rasterio.features import rasterize
from skimage.transform import resize
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from scipy.ndimage import distance_transform_edt

excluir_tipos = ['Cementerio']

# ============================================================
# PROCESAR CADA MUNICIPIO
# ============================================================
codigos = [c for c in gdf_dilig['Cod_Mun'].unique() if c != '000SD']
codigos = [str(c).zfill(5) for c in codigos]
codigos = list(dict.fromkeys(codigos))  # eliminar duplicados manteniendo orden

print(f"Códigos únicos: {len(codigos)}")
print(codigos[:5])

resultados = []
errores = []

for codigo in codigos:
    print(f"\nProcesando: {codigo}")
    
    try:
        # Geometría del municipio
        mun = municipios[municipios['MpCodigo'] == codigo]
        if len(mun) == 0:
            print(f"  ✗ Municipio no encontrado en shapefile")
            errores.append((codigo, 'municipio no encontrado'))
            continue
        
        geom = mun.geometry.values[0]
        bounds = mun.total_bounds  # minx, miny, maxx, maxy
        
        # Definir grid de referencia
        width  = max(int((bounds[2] - bounds[0]) / (RESOLUCION / 111320)), 10)
        height = max(int((bounds[3] - bounds[1]) / (RESOLUCION / 111320)), 10)
        transform = from_bounds(*bounds, width, height)
        shape = (height, width)

        # ── 1. NDVI ──────────────────────────────────────────
        ndvi_arr, _ = cargar_raster_municipio(PATH_NDVI, codigo, 'ndvi')
        if ndvi_arr is None:
            print(f"  ✗ NDVI no encontrado")
            errores.append((codigo, 'NDVI no encontrado'))
            continue
        # Redimensionar al grid de referencia
        from skimage.transform import resize
        ndvi_arr = resize(ndvi_arr, shape, anti_aliasing=True)
        print(f"  ✓ NDVI: {ndvi_arr.shape}")

        # ── 2. DEM → altitud y pendiente ────────────────────
        dem_arr, _ = cargar_raster_municipio(PATH_DEM, codigo, 'dem')
        if dem_arr is None:
            print(f"  ✗ DEM no encontrado")
            errores.append((codigo, 'DEM no encontrado'))
            continue
        dem_arr   = resize(dem_arr, shape, anti_aliasing=True)
        slope_arr = derivar_pendiente(dem_arr)
        print(f"  ✓ DEM + pendiente: {dem_arr.shape}")

        # ── 3. Distancia a hidrografía ───────────────────────
        hidro_path = Path(PATH_HIDRO)
        hidro_files = list(hidro_path.glob(f"{codigo}_*.gpkg"))
        if hidro_files:
            hidro = gpd.read_file(hidro_files[0]).to_crs(CRS)
            dist_hidro = raster_distancia(hidro, transform, shape)
        else:
            dist_hidro = np.full(shape, np.nan)
            print(f"  ! Hidrografía no encontrada — usando NaN")
        print(f"  ✓ Distancia hidrografía")

        # ── 4. Distancia a vías ──────────────────────────────
        vias_osm_path = Path(PATH_VIAS_OSM) / f"{codigo}_vias.gpkg"
        if vias_osm_path.exists():
            vias = gpd.read_file(vias_osm_path).to_crs(CRS)
            dist_vias = raster_distancia(vias, transform, shape)
            print(f"  ✓ Distancia vías OSM: {len(vias)} segmentos")
        else:
            dist_vias = np.full(shape, np.nan)
            print(f"  ! Sin vías OSM para {codigo}")

        # ── 5. Distancia a centros poblados ──────────────────
        pob_path = Path(PATH_POBLADOS)
        pob_files = list(pob_path.glob(f"{codigo}_*.gpkg"))
        if pob_files:
            pob = gpd.read_file(pob_files[0]).to_crs(CRS)
            dist_pob = raster_distancia(pob, transform, shape)
        else:
            dist_pob = np.full(shape, np.nan)
            print(f"  ! Centros poblados no encontrados — usando NaN")
        print(f"  ✓ Distancia centros poblados")

        # ── 6. Distancia a lugares de referencia (EQUITAS) ───
        refs = gdf_dilig[
            (gdf_dilig['Cod_Mun'] == codigo)
        ].copy()
        if len(refs) > 0:
            dist_refs = raster_distancia(refs, transform, shape)
        else:
            dist_refs = np.full(shape, np.nan)
        print(f"  ✓ Distancia lugares de referencia")

        # ── 7. Variable de resultado (presencias) ────────────
        presencias = gdf_dilig[
        (gdf_dilig['Cod_Mun'] == codigo) & 
        (gdf_dilig['label'] == 1.0) &
        (~gdf_dilig['Tipo disposición (Cuando aplique)'].isin(excluir_tipos))
        ].copy()
        ausencias = gdf_dilig[
            (gdf_dilig['Cod_Mun'] == codigo) & 
            (gdf_dilig['label'] == 0.0)
        ].copy()

        # ── 8. Guardar stack de covariables ──────────────────
        stack = np.stack([
            ndvi_arr,
            dem_arr,
            slope_arr,
            dist_hidro,
            dist_vias,
            dist_pob,
            dist_refs
        ], axis=0)  # shape: (7, height, width)

        output_path = Path(PATH_OUTPUT) / f"covariables_{codigo}.npy"
        np.save(output_path, stack)

        # Guardar también las presencias y ausencias como CSV
        if len(presencias) > 0:
            presencias[['lon', 'lat', 'label']].to_csv(
                Path(PATH_OUTPUT) / f"presencias_{codigo}.csv", index=False
            )
        if len(ausencias) > 0:
            ausencias[['lon', 'lat', 'label']].to_csv(
                Path(PATH_OUTPUT) / f"ausencias_{codigo}.csv", index=False
            )

        resultados.append({
            'codigo': codigo,
            'n_presencias': len(presencias),
            'n_ausencias': len(ausencias),
            'shape': shape,
            'status': 'ok'
        })
        print(f"  ✓ Guardado: {output_path.name}")
        print(f"     Presencias: {len(presencias)}, Ausencias: {len(ausencias)}")

    except Exception as e:
        print(f"  ✗ Error: {e}")
        errores.append((codigo, str(e)))

In [ ]:
import rasterio
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm

X_list = []
y_list = []
grupos = []

nombres_covariables = [
    'ndvi', 'altitud', 'pendiente',
    'dist_hidrografia', 'dist_vias', 
    'dist_centros_poblados', 'dist_referencias'
]

def extraer_valor_raster(raster_path, lon, lat):
    """Extrae valor de un raster en una coordenada específica."""
    try:
        with rasterio.open(raster_path) as src:
            val = list(src.sample([(lon, lat)]))[0][0]
            if val == src.nodata:
                return np.nan
            return float(val)
    except:
        return np.nan

def calcular_pendiente_punto(dem_path, lon, lat, delta=0.001):
    """Calcula pendiente en un punto a partir del DEM."""
    try:
        with rasterio.open(dem_path) as src:
            z = list(src.sample([(lon, lat)]))[0][0]
            z_e = list(src.sample([(lon+delta, lat)]))[0][0]
            z_n = list(src.sample([(lon, lat+delta)]))[0][0]
            # Aproximación del gradiente
            dz_dx = (z_e - z) / (delta * 111320)
            dz_dy = (z_n - z) / (delta * 111320)
            pendiente = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))
            return float(pendiente)
    except:
        return np.nan

print("Cargando datos con extracción directa...")

for pres_path in tqdm(sorted(Path(PATH_PROCESSED).glob("presencias_*.csv")),
                      desc="Municipios"):
    codigo = pres_path.stem.replace("presencias_", "")

    # Paths de rasters
    ndvi_path   = Path("../data/landsat/landsat_ndvi/") / f"ndvi_{codigo}.tif"
    dem_path    = Path("../data/dem/dem_srtm/")  / f"dem_{codigo}.tif"
    hidro_path  = list(Path("../data/hidrografia/").glob(f"{codigo}_*.gpkg"))
    vias_path   = Path("../data/red_vial/RedVialOD/RedVial.shp")
    pob_path    = list(Path("../data/centros_poblados/").glob(f"{codigo}_*.gpkg"))
    refs_cov    = Path(PATH_PROCESSED) / f"covariables_{codigo}.npy"

    if not ndvi_path.exists() or not dem_path.exists():
        continue

    # Cargar presencias y ausencias
    dfs = []
    pres = pd.read_csv(pres_path)
    pres['label'] = 1
    dfs.append(pres)

    ause_path = Path(PATH_PROCESSED) / f"ausencias_{codigo}.csv"
    if ause_path.exists():
        ause = pd.read_csv(ause_path)
        ause['label'] = 0
        dfs.append(ause)

    puntos = pd.concat(dfs, ignore_index=True)
    if len(puntos) == 0:
        continue

    # Cargar dist_referencias del stack procesado (capa 6)
    stack_refs = None
    if refs_cov.exists():
        stack_refs = np.load(refs_cov)

        mun = municipios[municipios['MpCodigo'] == codigo]
        bounds = mun.total_bounds
        h, w = stack_refs.shape[1], stack_refs.shape[2]

    for _, punto in puntos.iterrows():
        lon, lat = punto['lon'], punto['lat']

        # Extraer cada covariable directamente del raster
        ndvi        = extraer_valor_raster(ndvi_path, lon, lat)
        altitud     = extraer_valor_raster(dem_path, lon, lat)
        pendiente   = calcular_pendiente_punto(dem_path, lon, lat)

        # Distancia a hidrografía — del stack procesado (capa 3)
        # Distancia a vías — del stack procesado (capa 4)
        # Distancia a centros poblados — del stack procesado (capa 5)
        # Distancia a referencias — del stack procesado (capa 6)
        if stack_refs is not None:
            col = int((lon - bounds[0]) / (bounds[2] - bounds[0]) * w)
            row = int((bounds[3] - lat) / (bounds[3] - bounds[1]) * h)
            col = np.clip(col, 0, w-1)
            row = np.clip(row, 0, h-1)
            dist_hidro  = stack_refs[3, row, col]
            dist_vias   = stack_refs[4, row, col]
            dist_pob    = stack_refs[5, row, col]
            dist_refs   = stack_refs[6, row, col]
        else:
            dist_hidro = dist_vias = dist_pob = dist_refs = np.nan

        covs = np.array([ndvi, altitud, pendiente,
                         dist_hidro, dist_vias, dist_pob, dist_refs])

        # Imputar NaN individuales con 0 por ahora
        covs = np.where(np.isnan(covs), 0.0, covs)

        X_list.append(covs)
        y_list.append(punto['label'])
        grupos.append(codigo)

X = np.array(X_list)
y = np.array(y_list)
grupos = np.array(grupos)

print(f"\n✓ Total puntos: {len(X)}")
print(f"  Presencias: {int(y.sum())}")
print(f"  Ausencias: {int((y==0).sum())}")
print(f"  Municipios: {len(np.unique(grupos))}")
print(f"  NaN en X: {np.isnan(X).sum()}")

# Verificar altitud y pendiente de Ituango
idx_it = grupos == '05361'
if idx_it.sum() > 0:
    print(f"\nItuango — primeros 3 puntos:")
    for i, row in enumerate(X[idx_it][:3]):
        print(f"  ndvi={row[0]:.3f}, altitud={row[1]:.0f}m, "
              f"pendiente={row[2]:.1f}°")

In [ ]:
print(f"Total puntos: {len(X)}")
print(f"Presencias: {int(y.sum())}")
print(f"Ausencias: {int((y==0).sum())}")
print(f"Municipios: {len(np.unique(grupos))}")
print(f"NaN en X: {np.isnan(X).sum()}")

# Verificar Ituango
idx_it = grupos == '05361'
if idx_it.sum() > 0:
    print(f"\nItuango — primeros 3 puntos:")
    for row in X[idx_it][:3]:
        print(f"  ndvi={row[0]:.3f}, alt={row[1]:.0f}m, pend={row[2]:.1f}°, vias={row[4]:.0f}m")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

logo = LeaveOneGroupOut()
resultados_rf = []

for train_idx, test_idx in logo.split(X, y, grupos):
    municipio_test = grupos[test_idx][0]
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    if y_test.sum() == 0:
        continue
    
    try:
        modelo_rf = RandomForestClassifier(
            n_estimators=200,
            max_depth=5,
            min_samples_leaf=5,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
        modelo_rf.fit(X_train, y_train)
        y_pred = modelo_rf.predict_proba(X_test)[:, 1]
        
        if len(np.unique(y_test)) > 1:
            auc = roc_auc_score(y_test, y_pred)
        else:
            auc = np.nan
        
        k = max(1, int(len(y_test) * 0.3))
        top_k = np.argsort(y_pred)[::-1][:k]
        prec_k = y_test[top_k].mean()
        
        resultados_rf.append({
            'municipio': municipio_test,
            'n_presencias_test': int(y_test.sum()),
            'auc': round(auc, 3) if not np.isnan(auc) else np.nan,
            'precision_at_k': round(prec_k, 3)
        })
        auc_str = f"{auc:.3f}" if not np.isnan(auc) else "N/A"
        print(f"  {municipio_test}: auc={auc_str}, "
              f"P@k={prec_k:.3f}")
        
    except Exception as e:
        print(f"  {municipio_test}: Error — {e}")



In [ ]:
df_rf = pd.DataFrame(resultados_rf)

print(f"\n{'='*50}")
print(f"Random Forest — todos los municipios:")
print(f"  AUC promedio: {df_rf['auc'].mean():.3f}")
print(f"  Precision@k promedio: {df_rf['precision_at_k'].mean():.3f}")

df_rf_rob = df_rf[df_rf['n_presencias_test'] >= 10]
print(f"\nRandom Forest — municipios >=10 presencias (n={len(df_rf_rob)}):")
print(f"  AUC promedio: {df_rf_rob['auc'].mean():.3f}")
print(f"  Precision@k promedio: {df_rf_rob['precision_at_k'].mean():.3f}")

print(f"\nComparación:")
print(f"{'Modelo':<15} {'AUC':>8} {'P@k':>8}")
print(f"{'Maxent':<15} {'0.539':>8} {'0.884':>8}")
print(f"{'Random Forest':<15} {df_rf['auc'].mean():>8.3f} {df_rf['precision_at_k'].mean():>8.3f}")

In [ ]:
from sklearn.model_selection import train_test_split

# Split 80-20 aleatorio sin distinguir municipio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} puntos ({y_train.sum():.0f} presencias)")
print(f"Test:  {len(X_test)} puntos ({y_test.sum():.0f} presencias)")


# Random Forest
modelo_rf2 = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
modelo_rf2.fit(X_train, y_train)
y_pred_rf = modelo_rf2.predict_proba(X_test)[:, 1]

auc_rf = roc_auc_score(y_test, y_pred_rf)
top_k_rf = np.argsort(y_pred_rf)[::-1][:k]
prec_k_rf = y_test[top_k_rf].mean()

print(f"\nComparación 80-20 split:")
print(f"{'Modelo':<15} {'AUC':>8} {'P@k':>8}")
print(f"{'Random Forest':<15} {auc_rf:>8.3f} {prec_k_rf:>8.3f}")

In [ ]:
modelo_rf_final = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

modelo_rf_final.fit(X, y)
print("✓ Modelo RF final entrenado")

# Guardar
import joblib
joblib.dump(modelo_rf_final, "../data/Processed/rf_modelo_final.pkl")
print("✓ Guardado")

In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
import rasterio
from rasterio.transform import from_bounds

codigo = '05361'
print(f"Generando mapa de probabilidades para Ituango ({codigo})...")

# Cargar geometría del municipio
mun = municipios[municipios['MpCodigo'] == codigo]
bounds = mun.total_bounds  # minx, miny, maxx, maxy

# Cargar rasters
ndvi_path = Path("../data/landsat/landsat_ndvi/") / f"ndvi_{codigo}.tif"
dem_path  = Path("../data/dem/dem_srtm/") / f"dem_{codigo}.tif"

# Definir grid de predicción
resolucion_grados = 30 / 111320
width  = int((bounds[2] - bounds[0]) / resolucion_grados)
height = int((bounds[3] - bounds[1]) / resolucion_grados)

# Limitar tamaño para no saturar memoria
max_dim = 300
if width > max_dim or height > max_dim:
    factor = max(width, height) / max_dim
    width  = int(width / factor)
    height = int(height / factor)

print(f"Grid de predicción: {height} x {width}")

# Construir grid de coordenadas
lons = np.linspace(bounds[0], bounds[2], width)
lats = np.linspace(bounds[3], bounds[1], height)
lon_grid, lat_grid = np.meshgrid(lons, lats)

# Extraer covariables para cada punto del grid
print("Extrayendo covariables del grid...")
X_grid = np.zeros((height * width, 6))
X_grid[:] = np.nan

coords_flat = list(zip(lon_grid.flatten(), lat_grid.flatten()))

# NDVI
with rasterio.open(ndvi_path) as src:
    ndvi_vals = np.array([v[0] for v in src.sample(coords_flat)], dtype=float)
    ndvi_vals[ndvi_vals == src.nodata] = np.nan

# DEM
with rasterio.open(dem_path) as src:
    dem_vals = np.array([v[0] for v in src.sample(coords_flat)], dtype=float)
    dem_vals[dem_vals == src.nodata] = np.nan

# Pendiente aproximada
delta = 0.001
coords_e = [(lon + delta, lat) for lon, lat in coords_flat]
coords_n = [(lon, lat + delta) for lon, lat in coords_flat]
with rasterio.open(dem_path) as src:
    dem_e = np.array([v[0] for v in src.sample(coords_e)], dtype=float)
    dem_n = np.array([v[0] for v in src.sample(coords_n)], dtype=float)

dz_dx = (dem_e - dem_vals) / (delta * 111320)
dz_dy = (dem_n - dem_vals) / (delta * 111320)
slope_vals = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))

# Cargar stack para distancias
stack = np.load(Path(PATH_PROCESSED) / f"covariables_{codigo}.npy")
h_stack, w_stack = stack.shape[1], stack.shape[2]

def coord_a_pixel(lon, lat):
    col = int((lon - bounds[0]) / (bounds[2] - bounds[0]) * w_stack)
    row = int((bounds[3] - lat) / (bounds[3] - bounds[1]) * h_stack)
    return np.clip(row, 0, h_stack-1), np.clip(col, 0, w_stack-1)

dist_hidro = np.array([
    stack[3, *coord_a_pixel(lon, lat)] 
    for lon, lat in coords_flat
])
dist_vias = np.array([
    stack[4, *coord_a_pixel(lon, lat)] 
    for lon, lat in coords_flat
])
dist_pob = np.array([
    stack[5, *coord_a_pixel(lon, lat)] 
    for lon, lat in coords_flat
])
dist_refs = np.array([
    stack[6, *coord_a_pixel(lon, lat)] 
    for lon, lat in coords_flat
])

# Ensamblar covariables — 7 columnas
X_grid = np.zeros((height * width, 7))
X_grid[:] = np.nan
X_grid[:, 0] = ndvi_vals
X_grid[:, 1] = dem_vals
X_grid[:, 2] = slope_vals
X_grid[:, 3] = dist_hidro
X_grid[:, 4] = dist_vias
X_grid[:, 5] = dist_pob
X_grid[:, 6] = dist_refs

# Imputar NaN con mediana del municipio en entrenamiento
idx_mun = grupos == codigo
X_mun = X[idx_mun]
for i in range(X_grid.shape[1]):
    nan_mask = np.isnan(X_grid[:, i])
    if nan_mask.sum() > 0:
        X_grid[nan_mask, i] = np.nanmedian(X_mun[:, i])

# Predecir
print("Prediciendo probabilidades...")
pi_flat = modelo_rf_final.predict_proba(X_grid)[:, 1]
pi_grid = pi_flat.reshape(height, width)

# Máscara del municipio
from rasterio.features import geometry_mask
from rasterio.transform import from_bounds as fb

transform = fb(bounds[0], bounds[1], bounds[2], bounds[3], width, height)
mask = geometry_mask(
    [mun.geometry.values[0].__geo_interface__],
    transform=transform,
    invert=True,
    out_shape=(height, width)
)
pi_grid[~mask] = np.nan

# ============================================================
# MAPA
# ============================================================
verde       = '#1B5E3B'
dorado      = '#C8A951'
verde_claro = '#74C69D'

fig, ax = plt.subplots(figsize=(10, 10))
ax.set_facecolor('#E8F4F8')
fig.patch.set_facecolor('white')

# Mapa de calor
cmap = mcolors.LinearSegmentedColormap.from_list(
    'fosas', ['#F5F5F0', '#97C98F', '#4A8C5C', '#1B5E3B', '#0A2E1A']
)

im = ax.imshow(
    pi_grid,
    extent=[bounds[0], bounds[2], bounds[1], bounds[3]],
    cmap=cmap, vmin=0, vmax=1,
    origin='upper', alpha=0.85
)

# Contorno del municipio
mun.boundary.plot(ax=ax, color='#333', linewidth=1.5)

# Puntos de presencias conocidas
pres = gpd.GeoDataFrame(
    pd.read_csv(Path(PATH_PROCESSED) / f"presencias_{codigo}.csv"),
    geometry=gpd.points_from_xy(
        pd.read_csv(Path(PATH_PROCESSED) / f"presencias_{codigo}.csv")['lon'],
        pd.read_csv(Path(PATH_PROCESSED) / f"presencias_{codigo}.csv")['lat']
    ),
    crs="EPSG:4326"
)
pres.plot(ax=ax, color='red', markersize=25, 
          alpha=0.9, zorder=5, label='Exhumaciones confirmadas')

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Probabilidad de presencia $\\hat{\\pi}_i$', 
               fontsize=10, color='#333')
cbar.ax.tick_params(colors='#666')

# Títulos y leyenda
ax.set_title(
    'Mapa de calor — Ituango, Antioquia\nModelo Random Forest',
    fontsize=13, pad=15, color='#1a1a1a'
)
ax.legend(fontsize=10, loc='lower right', framealpha=0.95)
ax.set_xlabel('Longitud', fontsize=10, color='#555')
ax.set_ylabel('Latitud', fontsize=10, color='#555')

ax.text(0.01, 0.01,
        'Fuente: EQUITAS Colombia. Modelo: Random Forest. Elaboración propia.',
        transform=ax.transAxes, fontsize=7.5, color='#999', style='italic')

plt.tight_layout()
plt.savefig('../data/Processed/mapa_calor_ituango_RF.png',
            dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print("✓ Mapa guardado")

In [ ]:
# Excluir dist_referencias (índice 6)
indices_usar = [0, 1, 2, 3, 4, 5]  # sin dist_referencias
X_sin_refs = X[:, indices_usar]

nombres_sin_refs = [
    'ndvi', 'altitud', 'pendiente',
    'dist_hidrografia', 'dist_vias',
    'dist_centros_poblados'
]

logo = LeaveOneGroupOut()
resultados_mx_sr = []
resultados_rf_sr = []

for train_idx, test_idx in logo.split(X_sin_refs, y, grupos):
    municipio_test = grupos[test_idx][0]
    X_train, X_test = X_sin_refs[train_idx], X_sin_refs[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    if y_test.sum() == 0:
        continue
    

    # Random Forest
    try:
        modelo_rf = RandomForestClassifier(
            n_estimators=200,
            max_depth=5,
            min_samples_leaf=5,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
        modelo_rf.fit(X_train, y_train)
        y_pred_rf = modelo_rf.predict_proba(X_test)[:, 1]
        
        auc_rf = roc_auc_score(y_test, y_pred_rf) if len(np.unique(y_test)) > 1 else np.nan
        prec_k_rf = y_test[np.argsort(y_pred_rf)[::-1][:k]].mean()
        
        resultados_rf_sr.append({
            'municipio': municipio_test,
            'n_presencias_test': int(y_test.sum()),
            'auc': round(auc_rf, 3) if not np.isnan(auc_rf) else np.nan,
            'precision_at_k': round(prec_k_rf, 3)
        })
    except Exception as e:
        print(f"  RF {municipio_test}: {e}")

df_rf_sr = pd.DataFrame(resultados_rf_sr)

print(f"\nComparación — sin dist_referencias:")
print(f"{'Modelo':<15} {'AUC':>8} {'P@k':>8}")
#print(f"{'Maxent':<15} {df_mx_sr['auc'].mean():>8.3f} {df_mx_sr['precision_at_k'].mean():>8.3f}")
print(f"{'Random Forest':<15} {df_rf_sr['auc'].mean():>8.3f} {df_rf_sr['precision_at_k'].mean():>8.3f}")

print(f"\nComparación — con dist_referencias (anterior):")
print(f"{'Modelo':<15} {'AUC':>8} {'P@k':>8}")
#print(f"{'Maxent':<15} {'0.539':>8} {'0.884':>8}")
print(f"{'Random Forest':<15} {'0.547':>8} {'0.861':>8}")

## XGBOOST

In [ ]:
# Instalar xgboost si no está
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'xgboost'], 
               capture_output=True)

from xgboost import XGBClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

# X sin dist_referencias
indices_usar = [0, 1, 2, 3, 4, 5]
X_6 = X[:, indices_usar]

logo = LeaveOneGroupOut()
resultados_xgb = []

for train_idx, test_idx in logo.split(X_6, y, grupos):
    municipio_test = grupos[test_idx][0]
    X_train, X_test = X_6[train_idx], X_6[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    if y_test.sum() == 0:
        continue
    
    try:
        # Calcular scale_pos_weight para clases desbalanceadas
        scale = (y_train == 0).sum() / (y_train == 1).sum()
        
        modelo_xgb = XGBClassifier(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            scale_pos_weight=scale,
            random_state=42,
            n_jobs=-1,
            verbosity=0,
            eval_metric='auc'
        )
        modelo_xgb.fit(X_train, y_train)
        y_pred = modelo_xgb.predict_proba(X_test)[:, 1]
        
        auc = roc_auc_score(y_test, y_pred) if len(np.unique(y_test)) > 1 else np.nan
        k = max(1, int(len(y_test) * 0.3))
        prec_k = y_test[np.argsort(y_pred)[::-1][:k]].mean()
        
        resultados_xgb.append({
            'municipio': municipio_test,
            'n_presencias_test': int(y_test.sum()),
            'auc': round(auc, 3) if not np.isnan(auc) else np.nan,
            'precision_at_k': round(prec_k, 3)
        })
        
    except Exception as e:
        print(f"  {municipio_test}: {e}")

df_xgb = pd.DataFrame(resultados_xgb)

print(f"\nResumen comparación — CV leave-one-out:")
print(f"{'Modelo':<15} {'AUC':>8} {'P@k':>8}")
#print(f"{'Maxent':<15} {'0.488':>8} {'0.872':>8}")
print(f"{'Random Forest':<15} {'0.564':>8} {'0.812':>8}")
print(f"{'XGBoost':<15} {df_xgb['auc'].mean():>8.3f} {df_xgb['precision_at_k'].mean():>8.3f}")

# Detalle por municipio
print(f"\nXGBoost por municipio:")
print(df_xgb[['municipio', 'n_presencias_test', 'auc', 'precision_at_k']]
      .sort_values('precision_at_k', ascending=False).to_string(index=False))

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
from xgboost import XGBClassifier

logo = LeaveOneGroupOut()

params_grid = [
    {'max_depth': 2, 'learning_rate': 0.05, 'n_estimators': 100},
    {'max_depth': 2, 'learning_rate': 0.05, 'n_estimators': 200},
    {'max_depth': 3, 'learning_rate': 0.05, 'n_estimators': 200},
    {'max_depth': 3, 'learning_rate': 0.10, 'n_estimators': 200},
    {'max_depth': 4, 'learning_rate': 0.05, 'n_estimators': 200},
    {'max_depth': 3, 'learning_rate': 0.01, 'n_estimators': 500},
]

resultados_grid_xgb = []

for params in params_grid:
    aucs, prec_ks = [], []
    
    for train_idx, test_idx in logo.split(X_6, y, grupos):
        X_train, X_test = X_6[train_idx], X_6[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        if y_test.sum() == 0:
            continue
        
        try:
            scale = (y_train == 0).sum() / (y_train == 1).sum()
            modelo = XGBClassifier(
                n_estimators=params['n_estimators'],
                max_depth=params['max_depth'],
                learning_rate=params['learning_rate'],
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=scale,
                random_state=42,
                n_jobs=-1,
                verbosity=0,
                eval_metric='auc'
            )
            modelo.fit(X_train, y_train)
            y_pred = modelo.predict_proba(X_test)[:, 1]
            
            if len(np.unique(y_test)) > 1:
                aucs.append(roc_auc_score(y_test, y_pred))
            
            k = max(1, int(len(y_test) * 0.3))
            prec_ks.append(y_test[np.argsort(y_pred)[::-1][:k]].mean())
            
        except:
            pass
    
    resultado = {
        'max_depth': params['max_depth'],
        'learning_rate': params['learning_rate'],
        'n_estimators': params['n_estimators'],
        'auc': round(np.mean(aucs), 3) if aucs else np.nan,
        'prec_k': round(np.mean(prec_ks), 3) if prec_ks else np.nan,
    }
    resultados_grid_xgb.append(resultado)
    print(f"  depth={params['max_depth']} lr={params['learning_rate']} "
          f"n={params['n_estimators']}: "
          f"AUC={resultado['auc']:.3f}, P@k={resultado['prec_k']:.3f}")

df_grid_xgb = pd.DataFrame(resultados_grid_xgb)

print(f"\nMejor por AUC:")
best_auc = df_grid_xgb.loc[df_grid_xgb['auc'].idxmax()]
print(best_auc.to_string())

print(f"\nMejor por P@k:")
best_pk = df_grid_xgb.loc[df_grid_xgb['prec_k'].idxmax()]
print(best_pk.to_string())

In [ ]:
scale = (y == 0).sum() / (y == 1).sum()

modelo_xgb_final = XGBClassifier(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    eval_metric='auc'
)

modelo_xgb_final.fit(X_6, y)
print("✓ XGBoost final entrenado")

import joblib
joblib.dump(modelo_xgb_final, "../data/Processed/xgb_modelo_final.pkl")
print("✓ Guardado")

In [ ]:
import sys
sys.path.append('../src')
from maps import build_mapa_calor

In [ ]:

def build_mapa_calor(cod_mun, modelo_final, nom_modelo, X, grupos, municipios):
    codigo = cod_mun
    print(f"Generando mapa de probabilidades para {codigo}...")

    # Cargar geometría del municipio
    mun = municipios[municipios['MpCodigo'] == codigo]
    bounds = mun.total_bounds  # minx, miny, maxx, maxy

    # Cargar rasters
    ndvi_path = Path("../data/landsat/landsat_ndvi/") / f"ndvi_{codigo}.tif"
    dem_path  = Path("../data/dem/dem_srtm/") / f"dem_{codigo}.tif"

    # Definir grid de predicción
    resolucion_grados = 30 / 111320
    width  = int((bounds[2] - bounds[0]) / resolucion_grados)
    height = int((bounds[3] - bounds[1]) / resolucion_grados)

    # Limitar tamaño para no saturar memoria
    max_dim = 300
    if width > max_dim or height > max_dim:
        factor = max(width, height) / max_dim
        width  = int(width / factor)
        height = int(height / factor)

    print(f"Grid de predicción: {height} x {width}")

    # Construir grid de coordenadas
    lons = np.linspace(bounds[0], bounds[2], width)
    lats = np.linspace(bounds[3], bounds[1], height)
    lon_grid, lat_grid = np.meshgrid(lons, lats)

    # Extraer covariables para cada punto del grid
    print("Extrayendo covariables del grid...")
    X_grid = np.zeros((height * width, 6))
    X_grid[:] = np.nan

    coords_flat = list(zip(lon_grid.flatten(), lat_grid.flatten()))

    # NDVI
    with rasterio.open(ndvi_path) as src:
        ndvi_vals = np.array([v[0] for v in src.sample(coords_flat)], dtype=float)
        ndvi_vals[ndvi_vals == src.nodata] = np.nan

    # DEM
    with rasterio.open(dem_path) as src:
        dem_vals = np.array([v[0] for v in src.sample(coords_flat)], dtype=float)
        dem_vals[dem_vals == src.nodata] = np.nan

    # Pendiente aproximada
    delta = 0.001
    coords_e = [(lon + delta, lat) for lon, lat in coords_flat]
    coords_n = [(lon, lat + delta) for lon, lat in coords_flat]
    with rasterio.open(dem_path) as src:
        dem_e = np.array([v[0] for v in src.sample(coords_e)], dtype=float)
        dem_n = np.array([v[0] for v in src.sample(coords_n)], dtype=float)

    dz_dx = (dem_e - dem_vals) / (delta * 111320)
    dz_dy = (dem_n - dem_vals) / (delta * 111320)
    slope_vals = np.degrees(np.arctan(np.sqrt(dz_dx**2 + dz_dy**2)))

    # Cargar stack para distancias
    stack = np.load(Path(PATH_PROCESSED) / f"covariables_{codigo}.npy")
    h_stack, w_stack = stack.shape[1], stack.shape[2]

    def coord_a_pixel(lon, lat):
        col = int((lon - bounds[0]) / (bounds[2] - bounds[0]) * w_stack)
        row = int((bounds[3] - lat) / (bounds[3] - bounds[1]) * h_stack)
        return np.clip(row, 0, h_stack-1), np.clip(col, 0, w_stack-1)

    dist_hidro = np.array([
        stack[3, *coord_a_pixel(lon, lat)] 
        for lon, lat in coords_flat
    ])
    dist_vias = np.array([
        stack[4, *coord_a_pixel(lon, lat)] 
        for lon, lat in coords_flat
    ])
    dist_pob = np.array([
        stack[5, *coord_a_pixel(lon, lat)] 
        for lon, lat in coords_flat
    ])
    dist_refs = np.array([
        stack[6, *coord_a_pixel(lon, lat)] 
        for lon, lat in coords_flat
    ])

    # SIN dist_refs — solo 6 covariables
    X_grid = np.zeros((height * width, 6))
    X_grid[:] = np.nan
    X_grid[:, 0] = ndvi_vals
    X_grid[:, 1] = dem_vals
    X_grid[:, 2] = slope_vals
    X_grid[:, 3] = dist_hidro
    X_grid[:, 4] = dist_vias
    X_grid[:, 5] = dist_pob

    # Imputar NaN con mediana del municipio en entrenamiento
    idx_mun = grupos == codigo
    X_mun = X_6[idx_mun]  # ← X_6 sin dist_referencias
    for i in range(X_grid.shape[1]):
        nan_mask = np.isnan(X_grid[:, i])
        if nan_mask.sum() > 0:
            X_grid[nan_mask, i] = np.nanmedian(X_mun[:, i])

    # Predecir
    print("Prediciendo probabilidades...")
    pi_flat = modelo_xgb_final.predict_proba(X_grid)[:, 1]
    pi_grid = pi_flat.reshape(height, width)

    # Máscara del municipio
    from rasterio.features import geometry_mask
    from rasterio.transform import from_bounds as fb

    transform = fb(bounds[0], bounds[1], bounds[2], bounds[3], width, height)
    mask = geometry_mask(
        [mun.geometry.values[0].__geo_interface__],
        transform=transform,
        invert=True,
        out_shape=(height, width)
    )
    pi_grid[~mask] = np.nan

    # ============================================================
    # MAPA
    # ============================================================
    verde       = '#1B5E3B'
    dorado      = '#C8A951'
    verde_claro = '#74C69D'

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_facecolor('#E8F4F8')
    fig.patch.set_facecolor('white')

    # Mapa de calor
    cmap = mcolors.LinearSegmentedColormap.from_list(
        'fosas', ['#F5F5F0', '#97C98F', '#4A8C5C', '#1B5E3B', '#0A2E1A']
    )

    im = ax.imshow(
        pi_grid,
        extent=[bounds[0], bounds[2], bounds[1], bounds[3]],
        cmap=cmap, vmin=0, vmax=1,
        origin='upper', alpha=0.85
    )

    # Contorno del municipio
    mun.boundary.plot(ax=ax, color='#333', linewidth=1.5)

    # Puntos de presencias conocidas
    pres = gpd.GeoDataFrame(
        pd.read_csv(Path(PATH_PROCESSED) / f"presencias_{codigo}.csv"),
        geometry=gpd.points_from_xy(
            pd.read_csv(Path(PATH_PROCESSED) / f"presencias_{codigo}.csv")['lon'],
            pd.read_csv(Path(PATH_PROCESSED) / f"presencias_{codigo}.csv")['lat']
        ),
        crs="EPSG:4326"
    )
    pres.plot(ax=ax, color='red', markersize=25, 
            alpha=0.9, zorder=5, label='Exhumaciones confirmadas')

    # Colorbar
    cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label('Probabilidad de presencia $\\hat{\\pi}_i$', 
                fontsize=10, color='#333')
    cbar.ax.tick_params(colors='#666')

    # Títulos y leyenda
    ax.set_title(
        f'Mapa de calor — {codigo}\nModelo {nom_modelo}',
        fontsize=13, pad=15, color='#1a1a1a'
    )
    ax.legend(fontsize=10, loc='lower right', framealpha=0.95)
    ax.set_xlabel('Longitud', fontsize=10, color='#555')
    ax.set_ylabel('Latitud', fontsize=10, color='#555')

    ax.text(0.01, 0.01,
            f'Fuente: EQUITAS Colombia. Modelo: {nom_modelo}. Elaboración propia.',
            transform=ax.transAxes, fontsize=7.5, color='#999', style='italic')

    plt.tight_layout()
    plt.savefig(f'../data/Processed/mapa_calor_{codigo}_{nom_modelo}.png',
                dpi=200, bbox_inches='tight', facecolor='white')
    plt.show()
    print("✓ Mapa guardado")

In [ ]:
build_mapa_calor(cod_mun = '05361', modelo_final=modelo_xgb_final, nom_modelo= 'XGBoost', X=X_6, grupos=grupos, municipios=municipios)